In [7]:
from torchvision.datasets import MNIST

train_data = MNIST(root='./data', train=True, download=True)
test_data = MNIST(root='./data', train=False, download=True)

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt

from PIL import Image

MNIST_path = 'MNIST_folder'

In [ ]:
def save_data(root_dir, split_name, data):
    data_dir = os.path.join(root_dir, split_name)

    for i in range(10):
        os.makedirs(os.path.join(data_dir, str(i)), exist_ok=True)

    for i, (image, label) in enumerate(data):
        path = os.path.join(data_dir, str(label), f'img_{i}.png')
        image.save(path)


save_data(MNIST_path, 'train', train_data)
save_data(MNIST_path, 'test', test_data)

In [2]:
from typing import Any


class MNISTDataset(Dataset):
    def __init__(self, root_dir, split_name, transform=None):
        self.root_dir = root_dir
        self.split_name = split_name
        self.transform = transform

        self.images_path = []
        self.labels = []

        class_folders = os.listdir(os.path.join(root_dir, split_name))

        for class_folder in class_folders:
            class_path = os.path.join(root_dir, split_name, class_folder)
            images = os.listdir(class_path)

            for image in images:
                if image.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                    self.images_path.append(os.path.join(class_path, image))
                    self.labels.append(int(class_folder))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index) -> Any:
        image_path = self.images_path[index]
        label = self.labels[index]

        image = Image.open(image_path)

        if self.transform:
            image = self.transform(image)

        return image, label

In [3]:
train_dataset = MNISTDataset(MNIST_path, 'train', transform=transforms.ToTensor())
test_dataset = MNISTDataset(MNIST_path, 'test', transform=transforms.ToTensor())

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [11]:
class MyMNISTClassifier(nn.Module):
    def __init__(self, input_size, output_size):
        super(MyMNISTClassifier, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, output_size)

        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.dropout(x)
        x = self.relu(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout(x)
        
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [12]:
model = MyMNISTClassifier(input_size=28 * 28, output_size=10)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [13]:
from tqdm.auto import tqdm

epoches = 15


for epoch in range(epoches):
    model.train()
    loop = tqdm(train_dataloader, desc=f"Epoch [{epoch + 1}/{epoches}]")

    for images, labels in loop:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        loop.set_postfix(loss=f"{loss.item():.4f}")

    print(f'Epoch [{epoch + 1}/{epoches}], Loss: {loss.item():.4f}')

Epoch [1/15]:   0%|          | 0/938 [00:00<?, ?it/s]

Epoch [1/15], Loss: 0.0281


Epoch [2/15]:   0%|          | 0/938 [00:00<?, ?it/s]

Epoch [2/15], Loss: 0.0587


Epoch [3/15]:   0%|          | 0/938 [00:00<?, ?it/s]

Epoch [3/15], Loss: 0.0319


... (20 outputs left out)

Epoch [14/15]:   0%|          | 0/938 [00:00<?, ?it/s]

Epoch [14/15], Loss: 0.0001


Epoch [15/15]:   0%|          | 0/938 [00:00<?, ?it/s]

Epoch [15/15], Loss: 0.0003


In [14]:
model.eval()
with torch.no_grad():
    correct = 0
    total = 0

    for images, labels in test_dataloader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 99 %
